#   Extracción de datos SECOP II (Bronce)

**Taller ETL – Cubo SECOP**
**Autor:** Jurani Zabala Hernandez
**Objetivo:** Extraer la información de contratación pública desde la API oficial del SECOP II (SODA v3 / datos.gov.co) y depositarla, sin transformar, en el área **bronce** del *data lake* (HDFS), como insumo para los siguientes notebooks del taller.

**Alcance notebook BRONZE:**
1. Conexión a Spark con soporte HDFS/Hive de acuerdo a requerimientos solicitados
2. Extracción paginada y robusta desde la API pública del SECOP II, con manejo de errores y reintentos.
3. Persistencia cruda (sin transformar) en el data lake, área **bronce**, **página por página** (no se acumula el dataset completo en memoria).
4. Validación básica de la extracción (conteo de registros, esquema).

### 1. Configuración del entorno y sesión de Spark
 Se inicializa Spark apuntando al namenode definido en el docker-compose.yml del entorno contenerizado provisto en el taller, con soporte Hive habilitado (necesario para los pasos  posteriores).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit

spark = (
    SparkSession.builder
    .appName("SECOP_Extraccion_Bronce")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.sql.catalogImplementation", "hive")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")  # conversión pandas->Spark mucho más rápida
    .enableHiveSupport()
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✅ Sesión Spark inicializada:", spark.version)

26/08/24 00:05:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


✅ Sesión Spark inicializada: 3.1.2


### 2. Extracción desde la API SECOP II
Se utiliza la API SODA v3 del portal de datos abiertos de Colombia, tal como está enlazada en la guía del taller:

1. Dataset: SECOP II - Contratos Electrónicos (jbjy-vk9h) — dataset completo: ~5.98 millones de filas × 85 columnas.
2. Endpoint: https://www.datos.gov.co/api/v3/views/jbjy-vk9h/query.json
3. Límite real de la API: Socrata (SODA) impone un máximo de 50,000 registros por página, sin importar qué valor de pageSize se solicite. 
4. De acuerdo a lo anterior y para efectos de este taller se define PAGE_SIZE = 2000 y TOTAL_LIMIT = 20000 para que ejecute adecuadamente todos los pasos y limitar el volumen por razones de tiempo/recursos para que la  extracción esté bien implementada (paginación, manejo de errores, cobertura de fragmentos)
5. Diseño para no saturar la memoria del driver: con ~5.98M de filas y 85 columnas, acumular todo el dataset en una sola lista de Python antes de escribirlo puede agotar la memoria del contenedor de Jupyter (configurado con 4 GB para el driver de Spark). Por eso este notebook escribe cada página a HDFS inmediatamente después de descargarla, en vez de acumular todo y escribir al final. 

**Buenas prácticas aplicadas:**

1. Paginación completa mediante pageNumber / pageSize (máximo permitido por la API) hasta agotar los registros disponibles o el tope configurado en TOTAL_LIMIT.
2. Control robusto de errores: reintentos con backoff exponencial ante fallos de red o códigos HTTP distintos de 200.
3. Escritura incremental (página por página) para soportar volúmenes grandes sin agotar memoria.
4. Registro de metadatos de auditoría (fecha de extracción, página, tamaño de lote).

In [2]:
import requests
import time
import pandas as pd
from datetime import datetime

API_URL = "https://www.datos.gov.co/api/v3/views/jbjy-vk9h/query.json"
PAGE_SIZE = 2000         # Máximo permitido por la API SODA (no usar valores mayores)
TOTAL_LIMIT = 20000     # Tope de registros a extraer en esta corrida.
                          # El dataset completo tiene ~5,980,000 filas: si quieres extraerlo
                          # TODO, pon TOTAL_LIMIT = None (tomará varias horas por el volumen).
                          # Para desarrollo/pruebas del taller, un subconjunto como este es razonable
                          # y sigue cumpliendo con una extracción completa y correctamente paginada.
MAX_RETRIES = 3

def extraer_pagina(page_number: int, page_size: int = PAGE_SIZE, intentos: int = MAX_RETRIES):
    """Extrae una página de resultados desde la API SODA v3 (POST + SoQL) con reintentos y backoff exponencial."""
    body = {
        "query": "SELECT *",
        "page": {"pageNumber": page_number, "pageSize": page_size},
        "includeSynthetic": False,
    }
    for intento in range(1, intentos + 1):
        try:
            response = requests.post(
                API_URL,
                json=body,
                headers={"Content-Type": "application/json"},
                timeout=60,
            )
            if response.status_code == 200:
                return response.json()
            else:
                print(f" HTTP {response.status_code} en pageNumber={page_number} (intento {intento}/{intentos})")
        except requests.exceptions.RequestException as e:
            print(f" Error de red en pageNumber={page_number} (intento {intento}/{intentos}): {e}")
        time.sleep(2 ** intento)  # backoff exponencial: 2s, 4s, 8s...
    print(f" No se pudo extraer la página {page_number} tras {intentos} intentos.")
    return []

registros = []
page_number = 1
fecha_extraccion = datetime.utcnow().isoformat()

while len(registros) < TOTAL_LIMIT:
    print(f"Descargando página {page_number} (tamaño {PAGE_SIZE}) ...")
    batch = extraer_pagina(page_number, PAGE_SIZE)
    if not batch:
        # Página vacía o fallida: se detiene la extracción (no hay más datos o error persistente)
        break
    registros.extend(batch)
    page_number += 1

print(f"\n✅ Total de registros descargados: {len(registros)}")

Descargando página 1 (tamaño 2000) ...
Descargando página 2 (tamaño 2000) ...
Descargando página 3 (tamaño 2000) ...
Descargando página 4 (tamaño 2000) ...
Descargando página 5 (tamaño 2000) ...
Descargando página 6 (tamaño 2000) ...
Descargando página 7 (tamaño 2000) ...
Descargando página 8 (tamaño 2000) ...
Descargando página 9 (tamaño 2000) ...
Descargando página 10 (tamaño 2000) ...

✅ Total de registros descargados: 20000


## 3. Extracción y persistencia incremental (página por página)

En vez de acumular todas las páginas en memoria y escribir al final, cada página se convierte a DataFrame de Spark y se **agrega (`append`) inmediatamente** al área bronce. Esto permite procesar el dataset completo (millones de filas) sin necesidad de que quepa todo junto en la memoria del driver.

- Persistencia en el área Bronce del Data Lake
Los datos se guardan tal como llegan de la fuente (sin normalizar ni tipar), en formato Parquet, particionados por fecha de ingesta. Esto sigue el patrón medallion (bronce - plata - oro) solicitado para este taller:

Bronce (esta etapa): datos crudos, fieles a la fuente.
Plata: se genera en Transformacion.ipynb.
Oro: se genera en Cargue.ipynb, ya cargado en el cubo Hive.

In [3]:
from pyspark.sql.functions import to_date

BRONZE_PATH = "hdfs://namenode:9000/datalake/bronze/secop/contratos"
fecha_extraccion = datetime.utcnow().isoformat()

page_number = 1
total_extraidos = 0
primera_escritura = True  # controla si la primera página sobreescribe o agrega

while TOTAL_LIMIT is None or total_extraidos < TOTAL_LIMIT:
    print(f"Descargando página {page_number} (tamaño {PAGE_SIZE}) ...")
    batch = extraer_pagina(page_number, PAGE_SIZE)
    if not batch:
        # Página vacía o fallida: se detiene la extracción (no hay más datos o error persistente)
        break

    # Página -> pandas -> Spark (con Arrow habilitado), casteando a string para evitar
    # inferencias erróneas de tipo en bronce
    df_pagina = pd.DataFrame(batch).astype(str)
    df_pagina["_fecha_extraccion"] = fecha_extraccion
    df_pagina["_fuente"] = "SECOP_II_API_datos.gov.co"

    df_spark_pagina = (
        spark.createDataFrame(df_pagina)
        .coalesce(4)  # reduce el número de tareas paralelas -> tareas más livianas, escritura más rápida
        .withColumn("fecha_ingesta", to_date(lit(fecha_extraccion)))
    )

    modo = "overwrite" if primera_escritura else "append"
    df_spark_pagina.write.mode(modo).partitionBy("fecha_ingesta").parquet(BRONZE_PATH)
    primera_escritura = False

    total_extraidos += len(batch)
    print(f"  -> {len(batch)} registros escritos en bronce. Acumulado: {total_extraidos}")

    page_number += 1

print(f"\n✅ Extracción finalizada. Total de registros persistidos en bronce: {total_extraidos}")

Descargando página 1 (tamaño 2000) ...


26/08/24 00:05:46 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  -> 2000 registros escritos en bronce. Acumulado: 2000
Descargando página 2 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 4000
Descargando página 3 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 6000
Descargando página 4 (tamaño 2000) ...


...

  -> 2000 registros escritos en bronce. Acumulado: 10000
Descargando página 6 (tamaño 2000) ...
  -> 2000 registros escritos en bronce. Acumulado: 12000
Descargando página 7 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 14000
Descargando página 8 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 16000
Descargando página 9 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 18000
Descargando página 10 (tamaño 2000) ...


  -> 2000 registros escritos en bronce. Acumulado: 20000

✅ Extracción finalizada. Total de registros persistidos en bronce: 20000


## 3. Validación rápida de la extracción
Se revisa la forma de los datos crudos antes de persistirlos: número de columnas, muestra de filas y nulos evidentes.

In [4]:
df_pandas = pd.DataFrame(registros)
print(f"Filas: {len(df_pandas)} | Columnas: {len(df_pandas.columns)}")
df_pandas.head()

Filas: 20000 | Columnas: 85


,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,nombre_ordenador_de_pago,tipo_de_documento_ordenador_de_pago,n_mero_de_documento_ordenador_de_pago,documentos_tipo,descripcion_documentos_tipo,direcci_n_de_ejecuci_n_del_contrato,ultima_actualizacion,fecha_inicio_liquidacion,fecha_fin_liquidacion,fecha_de_notificaci_n_de_prorrogaci_n
0,ESE HOSPITAL DEPARTAMENTAL UNIVERSITARIO DEL Q...,800000118,Quindío,Armenia,"Colombia, Quindío , Armenia",Nacional,Salud y Protección Social,Corporación Autónoma,Centralizada,CO1.BDOS.3452554,...,No definido,No definido,No definido,No,No definido,AVENIDA BOLIVAR CALLE 17 NORTE\nArmenia\nQuind...,NaN,NaN,NaN,NaN
1,MUNICIPIO DE SOLEDAD,890106291,Atlántico,Soledad,"Colombia, Atlántico , Soledad",Territorial,Servicio Público,Ejecutivo,Centralizada,CO1.BDOS.9960799,...,EDISON MANUEL BARRERA REYES,Cédula de Ciudadanía,72231323,No,No definido,Calle 41 # 17-27 barrió la ilusión\nSoledad\nA...,2026-05-19T00:00:00.000,NaN,NaN,NaN
2,"Ministerio de las Culturas, las Artes y los Sa...",830034348,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",Nacional,Cultura,Ejecutivo,Centralizada,CO1.BDOS.8509712,...,No definido,No definido,No definido,No,No definido,Carrera 8 # 8 -55\nBogotá\nDistrito Capital de...,2026-03-31T00:00:00.000,2026-05-30T00:00:00.000,2026-11-30T00:00:00.000,NaN
3,HOSPITAL DEPARTAMENTAL SAN JUAN DE DIOS DE PUE...,842000004,Vichada,Puerto Carreño,"Colombia, Vichada , Puerto Carreño",Territorial,Salud y Protección Social,Corporación Autónoma,Centralizada,CO1.BDOS.9690037,...,No definido,No definido,No definido,No,No definido,Calle 36 N°40-89\nPuerto Carreño\nVichada\nCOL...,NaN,NaN,NaN,NaN
4,CONTRALORIA DEPARTAMENTAL DE NARIÑO,800157830,Nariño,No Definido,"Colombia, Nariño , No Definido",Territorial,No aplica/No pertenece,Corporación Autónoma,Descentralizada,CO1.BDOS.5042930,...,No definido,No definido,No definido,No,No definido,cra 24 #19-33 \nPasto\nNariño\nCOLOMBIA,2024-07-04T00:00:00.000,NaN,NaN,NaN


In [5]:
# Se agrega metadata de auditoría de la extracción antes de convertir a Spark
df_pandas["_fecha_extraccion"] = fecha_extraccion
df_pandas["_fuente"] = "SECOP_II_API_datos.gov.co"

df_raw = spark.createDataFrame(df_pandas.astype(str))  # se castea a string para evitar inferencias erróneas de tipo en el área bronce
df_raw.printSchema()
df_raw.show(5, truncate=True)

root
 |-- nombre_entidad: string (nullable = true)
 |-- nit_entidad: string (nullable = true)
 |-- departamento: string (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- localizaci_n: string (nullable = true)
 |-- orden: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- rama: string (nullable = true)
 |-- entidad_centralizada: string (nullable = true)
 |-- proceso_de_compra: string (nullable = true)
 |-- id_contrato: string (nullable = true)
 |-- referencia_del_contrato: string (nullable = true)
 |-- estado_contrato: string (nullable = true)
 |-- codigo_de_categoria_principal: string (nullable = true)
 |-- descripcion_del_proceso: string (nullable = true)
 |-- tipo_de_contrato: string (nullable = true)
 |-- modalidad_de_contratacion: string (nullable = true)
 |-- justificacion_modalidad_de: string (nullable = true)
 |-- fecha_de_firma: string (nullable = true)
 |-- fecha_de_inicio_del_contrato: string (nullable = true)
 |-- fecha_de_fin_del_contrato: string

26/08/24 00:06:32 WARN TaskSetManager: Stage 10 contains a task of very large size (4797 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+-----------+--------------------+--------------+--------------------+-----------+--------------------+--------------------+--------------------+-----------------+------------------+-----------------------+---------------+-----------------------------+-----------------------+--------------------+-------------------------+--------------------------+--------------------+----------------------------+-------------------------+----------------------+--------------------+-------------------+--------------------+--------+-------+------------------------+-----------+--------------------+------------------------+---------+----------------------+--------------+------------------+------------------------+---------------+-----------------------+------------+----------------+------------------+----------------------------+----------+--------------+---------------+----------------+------------------+-------------------+--------------------+--------------------------+------------

## 5. Verificación de la persistencia

In [6]:
df_check = spark.read.parquet(BRONZE_PATH)
print(f"Registros totales en bronce (todas las ingestas): {df_check.count()}")
df_check.select("id_contrato", "nombre_entidad", "fecha_ingesta").show(5, truncate=False)

Registros totales en bronce (todas las ingestas): 20000
+------------------+---------------------------------------+-------------+
|id_contrato       |nombre_entidad                         |fecha_ingesta|
+------------------+---------------------------------------+-------------+
|CO1.PCCNTR.4675714|DEPARTAMENTO DE POLICIA GUAINIA - DEGUN|2026-08-24   |
|CO1.PCCNTR.9795526|DEFENSORÍA DEL PUEBLO                  |2026-08-24   |
|CO1.PCCNTR.4384661|INSTITUCIÓN UNIVERSITARIA ITM          |2026-08-24   |
|CO1.PCCNTR.6694657|ALCALDIA LOCAL DE SAN CRISTOBAL        |2026-08-24   |
|CO1.PCCNTR.8560944|DEPARTAMENTO DE SANTANDER              |2026-08-24   |
+------------------+---------------------------------------+-------------+
only showing top 5 rows



## Conclusiones

- Se implementó la extracción completa desde la API pública del SECOP II mediante paginación y con LIMITE TOTAL de resgistros
- Se incorporó control robusto de errores con reintentos y backoff exponencial ante fallos de red o HTTP.
- Los datos crudos quedaron persistidos en el área **bronce** del *data lake* en formato Parquet, particionados por fecha de ingesta, listos para ser consumidos por `Transformacion.ipynb`.